# Notebook 02 — Landing Zone (CSV) → Bronze (Delta Lake)

Lê os arquivos CSV do bucket `landing-zone` do MinIO com **Apache Spark** e converte para o formato **Delta Lake** no bucket `bronze`.

**Fluxo:**
```
MinIO s3://landing-zone/<tabela>.csv
        ↓  Apache Spark 3.5.3 + Delta Lake 3.2.0
MinIO s3://bronze/<tabela>/  (Delta Table)
```

**O que é Delta Lake?**
> Delta Lake é uma camada de armazenamento open-source que traz transações ACID, versionamento e time travel para data lakes. Os dados são armazenados em Parquet com um transaction log (`_delta_log`) que registra cada operação.

## 1. Importações e Configuração

In [ ]:
import os
import boto3
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from botocore.client import Config
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip, DeltaTable

load_dotenv(find_dotenv())

MINIO_ENDPOINT       = os.getenv('MINIO_ENDPOINT',       'http://localhost:9020')
MINIO_ACCESS_KEY     = os.getenv('MINIO_ACCESS_KEY',     'minioadmin')
MINIO_SECRET_KEY     = os.getenv('MINIO_SECRET_KEY',     'minioadmin')
MINIO_LANDING_BUCKET = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')
MINIO_BRONZE_BUCKET  = os.getenv('MINIO_BRONZE_BUCKET',  'bronze')

print('Configuração carregada.')
print(f'  Landing : s3a://{MINIO_LANDING_BUCKET}/')
print(f'  Bronze  : s3a://{MINIO_BRONZE_BUCKET}/')

## 2. Criar SparkSession com suporte a Delta Lake e MinIO (S3A)

In [ ]:
builder = (
    SparkSession.builder
    .appName('LandingZone_to_Bronze_DeltaLake')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # Configuração S3A para MinIO
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.signing-algorithm', 'S3SignerType')
    # Pacotes Delta Lake + Hadoop AWS (jars baixados automaticamente pelo Maven)
    .config('spark.jars.packages',
            'io.delta:delta-spark_2.12:3.2.0,'
            'org.apache.hadoop:hadoop-aws:3.3.4,'
            'com.amazonaws:aws-java-sdk-bundle:1.12.262')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '4')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

print(f'Spark versão   : {spark.version}')
print(f'Delta versão   : {DeltaTable.__module__}')

## 3. Criar bucket `bronze` no MinIO (se não existir)

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

existing_buckets = [b['Name'] for b in s3.list_buckets().get('Buckets', [])]

if MINIO_BRONZE_BUCKET not in existing_buckets:
    s3.create_bucket(Bucket=MINIO_BRONZE_BUCKET)
    print(f'Bucket [{MINIO_BRONZE_BUCKET}] criado.')
else:
    print(f'Bucket [{MINIO_BRONZE_BUCKET}] já existe.')

## 4. Listar CSVs disponíveis no landing-zone

In [ ]:
response = s3.list_objects_v2(Bucket=MINIO_LANDING_BUCKET)
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no landing-zone:')
for f in csv_files:
    print(f'  - {f}')

## 5. Converter CSV → Delta Lake

Para cada CSV do `landing-zone`, o Spark lê o arquivo e escreve em formato Delta no bucket `bronze`.

In [ ]:
resultados = []

for csv_key in csv_files:
    tabela = csv_key.replace('.csv', '')
    
    src_path    = f's3a://{MINIO_LANDING_BUCKET}/{csv_key}'
    delta_path  = f's3a://{MINIO_BRONZE_BUCKET}/{tabela}'
    
    # Lê CSV
    df = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .option('encoding', 'UTF-8')
        .csv(src_path)
    )
    
    n_rows = df.count()
    
    # Escreve em Delta Lake
    (
        df.write
        .format('delta')
        .mode('overwrite')
        .save(delta_path)
    )
    
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    resultados.append({'tabela': tabela, 'registros': n_rows, 'delta_ok': is_delta, 'path': delta_path})
    print(f'  [{tabela}]  {n_rows} registros  Delta={is_delta}')

print(f'\nConversão concluída: {len(resultados)} tabelas Delta Lake criadas.')

## 6. Validar tabelas Delta — schema e amostra de dados

In [ ]:
for r in resultados:
    print(f"\n{'='*55}")
    print(f"Tabela : {r['tabela']}")
    print(f"Path   : {r['path']}")
    print(f"Linhas : {r['registros']}")
    
    df_delta = spark.read.format('delta').load(r['path'])
    print('Schema :')
    df_delta.printSchema()
    print('Amostra (3 linhas):')
    df_delta.show(3, truncate=False)

## 7. Inspecionar o Transaction Log Delta (_delta_log)

O `_delta_log` é o coração do Delta Lake — um diretório de arquivos JSON que registra cada operação de escrita (commit).

In [ ]:
# Inspeciona o transaction log da tabela 'apolice' como exemplo
tabela_exemplo = 'apolice'
delta_path_exemplo = f's3a://{MINIO_BRONZE_BUCKET}/{tabela_exemplo}'

dt = DeltaTable.forPath(spark, delta_path_exemplo)
print(f'History da tabela Delta [{tabela_exemplo}]:')
dt.history().select('version', 'timestamp', 'operation', 'operationParameters').show(truncate=False)

In [ ]:
spark.stop()
print('SparkSession encerrada.')